# **0) Installing Requirements + Project Setup**

In [1]:
!pip install gradio insightface --quiet # torch pre-installed
!git clone https://github.com/FireHead90544/EFNet/ && mv EFNet/* . && rm -r EFNet # EFNet core
!curl -L -o /content/ckpt_ep012.pth https://github.com/FireHead90544/EFNet/releases/download/modelweights/ckpt_ep012.pth # best checkpoint (ep_12)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 762.2/762.2 kB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 50.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 56.6 MB/s eta 0:00:00
Cloning into 'EFNet'...
remote: Enumerating objects: 15, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (15/15), done.
remote: Total 15 (delta 1), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (15/15), 44.60 KiB | 5.57 MiB/s, done.
Resolving deltas: 100% (1/1), done.
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0


In [2]:
# ============================================================
# Prepare Testing/Inference Showcase Dataset
# testfaces/ - 37 identities (10 images each, except testimages which have 7 each)
# testimages/ - 4 identities (3 imaages each)
# unknown/ - 3 identities (10 images each)
# ============================================================

!curl -L -o testfaces.zip "https://www.kaggle.com/api/v1/datasets/download/kasikrit/att-database-of-faces"
!mkdir testfaces && unzip -q testfaces.zip -d testfaces && rm testfaces.zip

from pathlib import Path
from PIL import Image
import shutil
import random

random.seed(42)  # reproducible split

# -----------------------------------------------------------------------------
# Convert ATT dataset -> testfaces/person_xxx/refN.jpg
# -----------------------------------------------------------------------------

src_root = Path("testfaces")

converted_root = Path("testfaces_converted")
converted_root.mkdir(exist_ok=True)

for subject_dir in sorted(src_root.glob("s*")):
    sid = int(subject_dir.name[1:])

    person_dir = converted_root / f"person_{sid:03d}"
    person_dir.mkdir(exist_ok=True)

    for pgm_file in sorted(
        subject_dir.glob("*.pgm"),
        key=lambda p: int(p.stem)
    ):
        img_num = int(pgm_file.stem)

        img = Image.open(pgm_file)

        out_file = person_dir / f"ref{img_num}.jpg"
        img.save(out_file, quality=95)

# Remove original extracted ATT folders
shutil.rmtree(src_root)

# Rename converted dataset -> testfaces
converted_root.rename("testfaces")

# -----------------------------------------------------------------------------
# Dataset split
# -----------------------------------------------------------------------------

gallery_root = Path("testfaces")      # enrollment/gallery
test_root = Path("testimages")        # closed-set testing
unknown_root = Path("unknown")        # open-set identities

test_root.mkdir(exist_ok=True)
unknown_root.mkdir(exist_ok=True)

all_ids = sorted(
    [p for p in gallery_root.iterdir() if p.is_dir()]
)

# ---------------------------------------------------------------------
# Pick 3 identities for open-set rejection
# ---------------------------------------------------------------------

unknown_ids = random.sample(all_ids, 3)

for person_dir in unknown_ids:
    shutil.move(
        str(person_dir),
        str(unknown_root / person_dir.name)
    )

remaining_ids = sorted(
    [p for p in gallery_root.iterdir() if p.is_dir()]
)

# ---------------------------------------------------------------------
# Pick 4 known identities for evaluation
# Remove 3 images from each and move to testimages
# ---------------------------------------------------------------------

eval_ids = random.sample(remaining_ids, 4)

for person_dir in eval_ids:

    dst_person = test_root / person_dir.name
    dst_person.mkdir(exist_ok=True)

    images = sorted(person_dir.glob("*.jpg"))

    # choose 3 images to hold out
    held_out = random.sample(images, 3)

    for img_path in held_out:
        shutil.move(
            str(img_path),
            str(dst_person / img_path.name)
        )

print("=" * 60)
print("Enrollment identities :", len(list(gallery_root.iterdir())))
print("Unknown identities    :", len(list(unknown_root.iterdir())))
print("Eval identities       :", len(eval_ids))
print("=" * 60)

print("\nOpen-set identities:")
for p in sorted(unknown_root.iterdir()):
    print(" ", p.name)

print("\nClosed-set eval identities:")
for p in sorted(eval_ids):
    print(" ", p.name)

print("\nDone.")

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 3693k  100 3693k    0     0  1559k      0  0:00:02  0:00:02 --:--:-- 2751k
Enrollment identities : 37
Unknown identities    : 3
Eval identities       : 4

Open-set identities:
  person_002
  person_008
  person_018

Closed-set eval identities:
  person_009
  person_011
  person_017
  person_019

Done.


# **1) Run Gradio Web Interface**

In [3]:
# ===========================
# GRADIO WEB DEMO
# ===========================

# @title Gradio Web Demo
# @markdown The default values should work fine for the time being.

import os
import torch
import gradio as gr
import numpy as np
from PIL import Image
import tempfile
from efnet.inference import EFNetInference

# ---------------------------------------------------------
# 1. Initialize Inference Engine
# ---------------------------------------------------------

MODEL_PATH = '/content/ckpt_ep012.pth' # @param {type: "string"}
DB_PATH = '/content/face_db.pth' # @param {type: "string"}
THRESHOLD = 0.622 # @param {type: "number"}

print("Initializing EFNet Engine...")
engine = EFNetInference(
    model_path=MODEL_PATH,
    threshold=THRESHOLD,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    use_aligner=True
)

# Load existing database if available
if os.path.exists(DB_PATH):
    engine.load_db(DB_PATH)
else:
    print(f"No existing DB found at {DB_PATH}. Starting fresh.")


# ---------------------------------------------------------
# 2. Define Core Functions
# ---------------------------------------------------------
def recognize_face(img_upload, img_webcam, img_path):
    img_to_process = None
    if img_webcam is not None:
        img_to_process = img_webcam
    elif img_upload is not None:
        img_to_process = img_upload
    elif img_path and os.path.exists(img_path):
        img_to_process = np.array(Image.open(img_path).convert('RGB'))

    if img_to_process is None:
        return None, "No valid image provided."

    pil_img = Image.fromarray(img_to_process).convert('RGB')

    # Optionally extract aligned image for visualization
    aligned_img = pil_img
    if engine.aligner is not None:
        aligned = engine.aligner(pil_img)
        if aligned is not None:
            aligned_img = aligned

    # Perform prediction on the aligned crop
    name, score = engine.predict(aligned_img)
    return np.array(aligned_img), f"Identity: {name} | Score: {score:.3f}"


def enroll_person(name, uploads, captured_imgs, folder_path):
    # 1. Bulk folder enrollment
    if folder_path and os.path.exists(folder_path):
        try:
            engine.enroll_from_folder(folder_path)
            return f"✅ Successfully enrolled all identities from folder: {folder_path}", captured_imgs, captured_imgs
        except Exception as e:
            return f"❌ Error enrolling from folder: {e}", captured_imgs, captured_imgs

    # 2. Individual enrollment
    if not name:
        return "⚠️ Please provide an identity name for manual enrollment.", captured_imgs, captured_imgs

    image_paths = []
    if uploads:
        # Gradio provides a list of File objects with .name attribute
        image_paths.extend([f.name for f in uploads])

    if captured_imgs:
        for img_array in captured_imgs:
            webcam_img = Image.fromarray(img_array)
            tmp = tempfile.NamedTemporaryFile(suffix=".jpg", delete=False)
            webcam_img.save(tmp.name)
            image_paths.append(tmp.name)

    if not image_paths:
        return "⚠️ No images provided for enrollment.", captured_imgs, captured_imgs

    try:
        engine.enroll(name, image_paths)
        # Clear the captured images on success
        return f"✅ Successfully enrolled '{name}' with {len(image_paths)} reference images!", [], []
    except Exception as e:
        return f"❌ Error during enrollment: {e}", captured_imgs, captured_imgs


def get_db_status():
    if not engine.prototypes:
        return "Database is empty. Enroll some identities first."
    status_str = f"**Total Enrolled Identities:** {len(engine.prototypes)}\n\n"
    for name, count in engine.ref_counts.items():
        status_str += f"- **{name}**: {count} reference images\n"
    return status_str

def save_db():
    os.makedirs(os.path.dirname(DB_PATH), exist_ok=True)
    engine.save_db(DB_PATH)
    return f"✅ Database saved to {DB_PATH}\n\n" + get_db_status()

def load_db():
    try:
        engine.load_db(DB_PATH)
        return f"✅ Database loaded from {DB_PATH}\n\n" + get_db_status()
    except Exception as e:
        return f"❌ Error loading database: {e}"

def unenroll_person(name):
    if not name:
        return "⚠️ Please enter a name to unenroll.", get_db_status()
    if name in engine.prototypes:
        engine.unenroll(name)
        return f"✅ '{name}' has been completely removed from the database.", get_db_status()
    else:
        return f"⚠️ '{name}' not found in the database.", get_db_status()

def update_threshold(val):
    engine.set_threshold(val)
    return f"Threshold updated to τ={val:.3f}"


# ---------------------------------------------------------
# 3. Build Gradio Interface
# ---------------------------------------------------------
with gr.Blocks(theme=gr.themes.Soft(primary_hue="blue")) as app:
    gr.Markdown("# 🧑‍💻 EFNet Edge Face Network Showcase")
    gr.Markdown("Real-time few-shot open-set face recognition system.")

    with gr.Tabs():
        # --- TAB 1: INFERENCE ---
        with gr.Tab("Live Recognition"):
            gr.Markdown("Upload an image, take a photo, or provide a path to test the recognition.")
            with gr.Row():
                with gr.Column():
                    infer_upload = gr.Image(label="Upload Image")
                    infer_webcam = gr.Image(sources=["webcam"], label="Webcam Capture")
                    infer_path = gr.Textbox(label="Or provide Image Path (e.g. /content/testimages/person_009/ref6.jpg)")
                    infer_btn = gr.Button("Recognize Face", variant="primary")
                with gr.Column():
                    infer_out_img = gr.Image(label="Aligned Face Preview")
                    infer_out_lbl = gr.Textbox(label="Prediction Result", placeholder="Result will appear here...")

            infer_btn.click(
                fn=recognize_face,
                inputs=[infer_upload, infer_webcam, infer_path],
                outputs=[infer_out_img, infer_out_lbl]
            )

        # --- TAB 2: ENROLLMENT ---
        with gr.Tab("Enroll Identity"):
            gr.Markdown("Register a new person to the face database. Provide multiple images for better accuracy.")

            # State to hold multiple webcam captures
            captured_images_state = gr.State([])

            with gr.Row():
                with gr.Column(scale=2):
                    gr.Markdown("### Manual Enrollment")
                    enroll_name = gr.Textbox(label="Identity Name (e.g. Alice)")
                    enroll_uploads = gr.File(label="Upload Reference Images", file_count="multiple")

                    gr.Markdown("#### Webcam Capture")
                    with gr.Row():
                        enroll_webcam = gr.Image(sources=["webcam"], label="Webcam Stream")
                        with gr.Column():
                            add_webcam_btn = gr.Button("➕ Add Frame to References", variant="secondary")
                            clear_webcam_btn = gr.Button("🗑️ Clear Captured Frames")
                            webcam_gallery = gr.Gallery(label="Captured Frames Ready for Enrollment", columns=2, height=200)

                    gr.Markdown("### Bulk Folder Enrollment")
                    enroll_folder = gr.Textbox(label="Folder Path (e.g. /content/testfaces)")

                    enroll_btn = gr.Button("🚀 Enroll Identity", variant="primary")
                with gr.Column(scale=1):
                    enroll_out_msg = gr.Textbox(label="Enrollment Status", lines=4)

            # Add webcam frame to state
            def add_frame(img, current_list):
                if img is not None:
                    current_list.append(img)
                return current_list, current_list

            add_webcam_btn.click(
                fn=add_frame,
                inputs=[enroll_webcam, captured_images_state],
                outputs=[captured_images_state, webcam_gallery]
            )

            # Clear frames
            def clear_frames():
                return [], []

            clear_webcam_btn.click(
                fn=clear_frames,
                outputs=[captured_images_state, webcam_gallery]
            )

            enroll_btn.click(
                fn=enroll_person,
                inputs=[enroll_name, enroll_uploads, captured_images_state, enroll_folder],
                outputs=[enroll_out_msg, captured_images_state, webcam_gallery]
            )

        # --- TAB 3: SETTINGS & DATABASE ---
        with gr.Tab("Database & Settings"):
            with gr.Row():
                with gr.Column():
                    gr.Markdown("### Recognition Threshold (τ)")
                    gr.Markdown("Higher = Stricter (more 'Unknowns'). Lower = Lenient (more false positives).")
                    thresh_slider = gr.Slider(minimum=0.1, maximum=0.9, value=engine.threshold, step=0.01, label="Cosine Threshold")
                    thresh_msg = gr.Textbox(label="Threshold Status")
                    thresh_slider.change(fn=update_threshold, inputs=thresh_slider, outputs=thresh_msg)

                with gr.Column():
                    gr.Markdown("### Database Management")
                    db_status = gr.Markdown(get_db_status())
                    with gr.Row():
                        save_btn = gr.Button("Save Database")
                        load_btn = gr.Button("Load Database")

                    gr.Markdown("#### Unenroll Identity")
                    with gr.Row():
                        unenroll_name = gr.Textbox(label="Identity Name to Remove", scale=2)
                        unenroll_btn = gr.Button("Unenroll", variant="stop", scale=1)

                    db_msg = gr.Textbox(label="Action Status", lines=5)

                    save_btn.click(fn=save_db, outputs=db_msg)
                    load_btn.click(fn=load_db, outputs=db_msg)
                    unenroll_btn.click(
                        fn=unenroll_person,
                        inputs=[unenroll_name],
                        outputs=[db_msg, db_status]
                    )

# ---------------------------------------------------------
# 4. Launch App
# ---------------------------------------------------------
# share=True creates a public link accessible from anywhere
app.launch(share=True, debug=True)

Initializing EFNet Engine...
download_path: /root/.insightface/models/buffalo_sc


100%|██████████| 14619/14619 [00:00<00:00, 62271.59KB/s]


Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_sc/det_500m.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_sc/w600k_mbf.onnx recognition ['None', 3, 112, 112] 127.5 127.5
set det-size: (320, 320)
[FaceAligner] InsightFace SCRFD loaded successfully.
[EFNetInference] Model loaded from '/content/ckpt_ep012.pth' on cuda. τ=0.622
No existing DB found at /content/face_db.pth. Starting fresh.


/tmp/ipykernel_2261/1208020813.py:141: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(primary_hue="blue")) as app:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://aba34f6460d42fb6db.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://aba34f6460d42fb6db.gradio.live
